# Project 2: Git Clone & Environment Setup

In [1]:
# 1. Clone or Pull Repository
%cd /content/

import os
REPO_URL = "https://github.com/cyunchaeskku/AILab_Project2.git"
REPO_NAME = "AILab_Project2"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest changes for {REPO_NAME}...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}

/content
Pulling latest changes for AILab_Project2...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 341 bytes | 341.00 KiB/s, done.
From https://github.com/cyunchaeskku/AILab_Project2
   de6fb57..19905cf  main       -> origin/main
Updating de6fb57..19905cf
Fast-forward
 configs/progressive_1024.yaml | 8 ++++----
 1 file changed, 4 insertions(+), 4 deletions(-)


In [2]:
# 2. Install dependencies
!pip install gdown
%cd AILab_Project2
!pip install -r requirements.txt

/content/AILab_Project2


In [4]:
# 3. Mount Google Drive for Checkpoints
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/OpenAILab/project2")
(DRIVE_PROJECT / "512").mkdir(parents=True, exist_ok=True)
(DRIVE_PROJECT / "1024").mkdir(parents=True, exist_ok=True)
!ls -lh /content/drive/MyDrive/OpenAILab/project2


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 12K
drwx------ 2 root root 4.0K May 28 07:46 1024
drwx------ 2 root root 4.0K May 27 15:44 512
drwx------ 2 root root 4.0K May 27 15:35 checkpoints


In [19]:
# 4. Download Train/Validation Datasets
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

files = {
    # Training data only. Use these for model training.
    "train_50k_256.zip": "1Bx2wH_ZJ44RXEY5X5pJJyYY8NaZbHa9w",
    "train_50k_512.zip": "1Sckvcw27OggN49FeOTN6_-xg-VZVNr8I",
    "train_50k_1024.zip": "1PhuCuv9cRNYUj1ItiSUUaX0_2zOBQjQs",

    # Validation data only. Do not use these for training.
    "valid_10k_256.zip": "1yEA26_FEgHwPur2jZCr0lmONesA6TyI2",
    "valid_10k_512.zip": "1hLDAyFgTZA2KRYSdJvPLhHJ3K0AgWfpE",
    "valid_10k_1024.zip": "1V0HfqDxUytt-wOiqyerXYC1-61EE8HqZ",
}

for filename, file_id in files.items():
    out = DATA_DIR / filename
    if out.exists() and out.stat().st_size > 0:
        print(f"skip {out} ({out.stat().st_size / 1024**3:.2f} GB)")
        continue
    print(f"download {out}")
    !gdown "https://drive.google.com/uc?id={file_id}" -O "{out}"

!ls -lh data


skip data/train_50k_256.zip (1.54 GB)
download data/train_50k_512.zip
Downloading...
From (original): https://drive.google.com/uc?id=1Sckvcw27OggN49FeOTN6_-xg-VZVNr8I
From (redirected): https://drive.google.com/uc?id=1Sckvcw27OggN49FeOTN6_-xg-VZVNr8I&confirm=t&uuid=47e05d8b-c226-425d-b968-111e70f6c009
To: /content/AILab_Project2/data/train_50k_512.zip
100% 5.57G/5.57G [00:46<00:00, 119MB/s] 
download data/train_50k_1024.zip
Downloading...
From (original): https://drive.google.com/uc?id=1PhuCuv9cRNYUj1ItiSUUaX0_2zOBQjQs
From (redirected): https://drive.google.com/uc?id=1PhuCuv9cRNYUj1ItiSUUaX0_2zOBQjQs&confirm=t&uuid=8c7225a4-da30-444a-8779-473fddf3d8b0
To: /content/AILab_Project2/data/train_50k_1024.zip
100% 18.2G/18.2G [03:15<00:00, 93.0MB/s]
skip data/valid_10k_256.zip (0.31 GB)
skip data/valid_10k_512.zip (1.04 GB)
skip data/valid_10k_1024.zip (3.38 GB)
total 29G
-rw-r--r-- 1 root root  17G May 15 03:58 train_50k_1024.zip
-rw-r--r-- 1 root root 1.6G May 15 03:47 train_50k_256.zip
-r

In [23]:
# 5. Verify Baseline Sample
!python generate.py --ckpt ckpt/ffhq256_baseline.pt --out sample_256.png --n 16

Architecture: build_baseline_256_generator() (no meta in ckpt)
Weights: G_ema_state  (21.20M params)
Saved 16 samples → sample_256.png


In [5]:
# 6. Login to Weights & Biases
import getpass
import os
import wandb

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY: ")

wandb.login(key=os.environ["WANDB_API_KEY"])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cyunchaeskku (cyunchaeskku-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# 7. Clean GPU Processes
import gc
import os
import signal
import subprocess

print("GPU processes before cleanup:")
!nvidia-smi

current_pid = os.getpid()
cmd = [
    "nvidia-smi",
    "--query-compute-apps=pid,process_name,used_memory",
    "--format=csv,noheader,nounits",
]
result = subprocess.run(cmd, capture_output=True, text=True)

for line in result.stdout.strip().splitlines():
    if not line.strip():
        continue
    pid_text, name, mem_text = [part.strip() for part in line.split(",", 2)]
    pid = int(pid_text)
    if pid == current_pid:
        print(f"keep current kernel pid={pid} mem={mem_text}MiB")
        continue
    print(f"kill gpu process pid={pid} name={name} mem={mem_text}MiB")
    os.kill(pid, signal.SIGTERM)

gc.collect()
try:
    import torch
    torch.cuda.empty|_cache()
except Exception as exc:
    print(f"torch cache cleanup skipped: {exc}")

print("GPU processes after cleanup:")
!nvidia-smi


GPU processes before cleanup:
Thu May 28 15:32:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0            290W /  400W |   26554MiB /  40960MiB |     98%      Default |
|                                         |                        |             Disabled |
+-----------------

In [ ]:
# 8. Start True Progressive Training from Professor 256 Baseline
# Uses PG-GAN-style fade-in schedule: 256 stabilize -> 512 fade/stabilize -> 1024 fade/stabilize.
# !python train.py --config configs/progressive_1024.yaml --init-from ckpt/ffhq256_baseline.pt --auto-resume
!python train.py --config configs/progressive_1024.yaml --resume runs/pggan_1024/ckpt_000050032.pt --new-wandb-run


Generator: 21.31M params
Discriminator: 20.25M params
Optimizers: G lr=0.0005, D lr=0.0005
Progressive schedule:
  #0 stabilize res=256 images=50000 batch=32 zip=data/train_50k_256.zip
  #1 fade res=512 images=300000 batch=16 zip=data/train_50k_512.zip
  #2 stabilize res=512 images=300000 batch=16 zip=data/train_50k_512.zip
  #3 fade res=1024 images=200000 batch=4 zip=data/train_50k_1024.zip
  #4 stabilize res=1024 images=500000 batch=4 zip=data/train_50k_1024.zip
Checkpoint backup dir: /content/drive/MyDrive/OpenAILab/project2/pggan_1024
Resuming from runs/pggan_1024/ckpt_000050032.pt
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cyunchaeskku (cyunchaeskku-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 1e04ufn1 (0.0s)
wandb: ⣻ setting up run 1e04ufn1 (0.0s)
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /conte

In [ ]:
# 9. Generate Samples from Progressive Checkpoint
!python generate.py --ckpt /content/drive/MyDrive/OpenAILab/project2/pggan_1024/final.pt --out progressive_samples.png --n 16 --nrow 4


In [ ]:
# 10. Export Progressive Generator to ONNX
!python export_onnx.py --ckpt /content/drive/MyDrive/OpenAILab/project2/pggan_1024/final.pt --out submission.onnx
